# GVH–CHH | Adjacent Orbital Subcube — Masked Reconstruction Gate FAST

**Baseline** `GVH_CHH_PROTOCOL_FROZEN_v1.0` (18 November 2026; unchanged).  
**Case** Jupiter: Io → **Europa withheld** → Ganymede, with Callisto as a negative comparison.  
**Scientific statuses** `CHH-STRUCTURE`, `INVERSE-INFERENCE`, `AUX-SENSITIVITY`, `KNOWN-PHYSICS`, `NUMERICAL-TEST`, `OPEN-LOCK`.  
**Not in scope**: `EOM-GVH`, `GVH Dynamic` action variation, empirical proof of a new gravitational constant, spectroscopic ALMA test (`GVH_USED=False`).

**Important anti-overclaim:** This is a **retrospective masked-record exercise**, *not* a historically blind discovery and *not* proof that an undetected moon must exist. The Io–Europa–Ganymede resonance is known already. Only the value for Europa is concealed from the model code until the evaluation cell.

**Authoritative data:** NASA/NSSDC [Jovian Satellite Fact Sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/joviansatfact.html) (periods and semi-major axes); NASA [Europa Facts](https://science.nasa.gov/jupiter/jupiter-moons/europa/europa-facts/) (Io–Europa–Ganymede near 1:2:4 resonance). Distances and periods are published rounded values, not instrument-level uncertainties.

## Mandatory four levels
1. **GVH/CHH candidate:** conditional orbital composition, unknown subcube candidate and residual degeneracies (no intrinsic global EOM).
2. **Established physics:** Keplerian periods and known Laplace mean-motion resonance; other model families are baselines.
3. **Analytical / numerical check:** freeze prediction before revealing Europa, null gap, synthetic false-positive, sensitivity, reproducibility.
4. **Scientific verdict:** assess accuracy of period estimate *separately* from detectability of a hidden moon and physical novelty.


## LEVEL 1 — Pre-specified model and decision gates

Conditionally assume that one *intermediate* orbital period in the **proposed** 2:1 × 2:1 resonant chain satisfies
\[
P_* = \sqrt{P_{\rm inner}P_{\rm outer}}.
\]

This interpolation is justified only under the chosen geometric-spacing model. It is not an EOM, discovery criterion, or universal CHH relation. Endpoint closeness to a factor of 4 is assessed using **a deliberately fixed demonstration threshold** `abs(P_out/P_in / 4 - 1) <= 0.05` (5%; arbitrary pedagogical choice, not empirical significance level). Do not tune after viewing withheld truth.

**Baselines:** `2 P_inner`, `P_outer/2` and linear midpoint `(P_inner+P_outer)/2`.  
**Decision rule:** ratio near 4 only opens a *candidate period*; output **ABSTAIN_BODY_EXISTENCE** without independent perturbation, transit, imaging, resonance-phase or other evidence. The same two measured endpoints are consistent with zero, one or several additional orbiting bodies.  
**Negative controls:** Ganymede–Callisto major-moon gap (not near 4); synthetic isolated pair with ratio 4 but explicitly no body (a false positive to any ratio-only existence claim).


In [1]:
from pathlib import Path
import hashlib, json, math, platform
from datetime import datetime, timezone
ROOT = Path('/mnt/data') if Path('/mnt/data').exists() else Path('.')
NAME = 'GVH_CHH_Adjacent_Orbital_Subcube_Blind_Reconstruction_Gate_FAST'
SOURCE_URL='https://nssdc.gsfc.nasa.gov/planetary/factsheet/joviansatfact.html'
RES_URL='https://science.nasa.gov/jupiter/jupiter-moons/europa/europa-facts/'
# Input record: NO Europa period or semi-major axis here.
observed = {
    'Io': {'period_d':1.769138, 'a_km':421800.0},
    'Ganymede': {'period_d':7.154553, 'a_km':1070400.0},
    'Callisto': {'period_d':16.689017, 'a_km':1882700.0},
}
threshold_fraction = 0.05   # pre-selected pedagogical gate, NOT calibrated probability
assert 0 < threshold_fraction < 1
checks=[]
def check(label, result, note=''):
    checks.append({'test':label,'pass':bool(result),'note':str(note)})
    if not result: raise AssertionError(f'{label}: {note}')

check('no_europa_in_observed_records', 'Europa' not in observed)
check('input_order_positive', 0 < observed['Io']['period_d'] < observed['Ganymede']['period_d'] < observed['Callisto']['period_d'])
check('source_url_declared', SOURCE_URL.startswith('https://nssdc.gsfc.nasa.gov/'))
print('Inputs frozen: Io, Ganymede, Callisto. Europa withheld until evaluation phase.')

Inputs frozen: Io, Ganymede, Callisto. Europa withheld until evaluation phase.


In [2]:
# Only observed endpoints are used to compute the prediction.
pi=observed['Io']['period_d']
pg=observed['Ganymede']['period_d']
pc=observed['Callisto']['period_d']
ratio=pg/pi
candidate_eligible=abs(ratio/4.-1.) <= threshold_fraction
predictions={
  'geometric_bridge_d':math.sqrt(pi*pg),
  'inner_times_two_d':2*pi,
  'outer_divided_two_d':pg/2,
  'arithmetic_midpoint_d':(pi+pg)/2,
  'semimajor_geometric_km':math.sqrt(observed['Io']['a_km']*observed['Ganymede']['a_km'])
}
null_ratio=pc/pg
null_gate=abs(null_ratio/4-1)<=threshold_fraction
# A synthetic pair exactly at 4:1, with zero true hidden bodies by construction.
synthetic_null={'p_inner_d':2.0,'p_outer_d':8.0,'true_hidden_count':0}
synthetic_ratio=synthetic_null['p_outer_d']/synthetic_null['p_inner_d']
synthetic_gate=abs(synthetic_ratio/4-1)<=threshold_fraction

pre_reveal={
  'protocol':'GVH_CHH_PROTOCOL_FROZEN_v1.0', 'test':'retrospective_masked_period',
  'sources':[SOURCE_URL,RES_URL], 'used_only':['Io','Ganymede','Callisto'],
  'fixed_gate_fraction':threshold_fraction, 'endpoint_ratio':ratio,
  'candidate_period_eligible':candidate_eligible, 'predictions':predictions,
  'negative_real_gap':{'pair':['Ganymede','Callisto'],'ratio':null_ratio,'four_to_one_gate':null_gate},
  'negative_synthetic_pair':{'ratio':synthetic_ratio,'has_hidden_body':False,'four_to_one_gate':synthetic_gate},
  'body_existence_verdict':'ABSTAIN_BODY_EXISTENCE',
  'GVH_USED':False
}
canonical=json.dumps(pre_reveal,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()
pre_reveal_sha256=hashlib.sha256(canonical).hexdigest()
(ROOT/(NAME+'_PRE_REVEAL_LOCK.json')).write_text(json.dumps({'prediction':pre_reveal,'prediction_sha256':pre_reveal_sha256},indent=2,ensure_ascii=False)+'\n')
check('positive_gap_approx_four_to_one',candidate_eligible)
check('negative_gap_rejected',not null_gate)
check('synthetic_null_exposes_false_positive',synthetic_gate and synthetic_null['true_hidden_count']==0)
check('period_prediction_between_endpoints',pi<predictions['geometric_bridge_d']<pg)
check('no_true_hidden_count_claim',pre_reveal['body_existence_verdict']=='ABSTAIN_BODY_EXISTENCE')
print('PRE-REVEAL prediction locked, SHA-256:',pre_reveal_sha256)
print('Geometric period prediction:', round(predictions['geometric_bridge_d'],8),'days')
print('Observed Io/Ganymede ratio:',round(ratio,6),' candidate gate:',candidate_eligible)
print('Ganymede/Callisto negative gate:',null_gate)
print('Synthetic 4:1 pair yields FALSE hidden-body trigger if gate misused:',synthetic_gate)

PRE-REVEAL prediction locked, SHA-256: 4a24fcbbf409188f7e2799442cd4d7b97befdebefc34c8cb333fb1b04bf7b1dc
Geometric period prediction: 3.55772281 days
Observed Io/Ganymede ratio: 4.04409  candidate gate: True
Ganymede/Callisto negative gate: False
Synthetic 4:1 pair yields FALSE hidden-body trigger if gate misused: True


## LEVEL 2 — What the formula does and does not imply

- A period prediction for a **hypothetical intermediate moon** is not evidence that the moon exists.
- The same endpoints support `H0` (no middle body), `H1` (one), `H2` (two or more) if only periods at the endpoints are known. The number and masses of unseen subcubes are **non-identifiable** without new independent observables.
- Geometric interpolation and 2:1–2:1 resonance are parts of *established* mathematical/astronomical modeling, not novel GVH dynamics.
- `P*` sensitivity gradients w.r.t. the two endpoint periods are **AUX-SENSITIVITY** (partial derivatives in parameter space), not temporal dynamics and not a new Einstein–æther coupling.


In [3]:
# AUX-SENSITIVITY: analytical vs finite-difference derivatives of P*=sqrt(Pi*Pg)
def bridge(a,b):
    return math.sqrt(a*b)
d_pi=0.5*math.sqrt(pg/pi)
d_pg=0.5*math.sqrt(pi/pg)
h=1e-5
fd_pi=(bridge(pi+h,pg)-bridge(pi-h,pg))/(2*h)
fd_pg=(bridge(pi,pg+h)-bridge(pi,pg-h))/(2*h)
elasticity_i=(pi/predictions['geometric_bridge_d'])*d_pi
elasticity_g=(pg/predictions['geometric_bridge_d'])*d_pg
check('sensitivity_analytic_numeric_io',abs(fd_pi-d_pi)<1e-9)
check('sensitivity_analytic_numeric_ganymede',abs(fd_pg-d_pg)<1e-9)
check('elasticity_is_one_half_each',abs(elasticity_i-0.5)<1e-12 and abs(elasticity_g-0.5)<1e-12)
print('AUX-SENSITIVITY dP*/dPi, dP*/dPg:',round(d_pi,9),round(d_pg,9),' elasticities=',elasticity_i,elasticity_g)
# ±1% endpoint perturbation is stress-testing input robustness, not reported measurement uncertainty.
stress_low=bridge(pi*0.99,pg*0.99)
stress_high=bridge(pi*1.01,pg*1.01)
check('stress_monotonic',stress_low<predictions['geometric_bridge_d']<stress_high)
print('Hypothetical ±1% endpoint stress gives period range',round(stress_low,6),round(stress_high,6),'days')

AUX-SENSITIVITY dP*/dPi, dP*/dPg: 1.005496125 0.248633479  elasticities= 0.5 0.5
Hypothetical ±1% endpoint stress gives period range 3.522146 3.5933 days


## LEVEL 3 — Reveal the withheld reference *only now*

The following cell contains published Europa values. It MUST remain below the frozen-prediction cell above. This test's evaluation is not historically independent: the resonance model was chosen based on well-known Galilean dynamics.


In [4]:
# PHASE 2: revealed reference for score computation, never passed to the phase 1 predictor.
reference_heldout = {'name':'Europa','period_d':3.551181,'a_km':671100.0,'source':SOURCE_URL}
truth=reference_heldout['period_d']
period_errors={k:{'predicted_d':v,'absolute_error_d':abs(v-truth),'percent_error':100*abs(v-truth)/truth}
               for k,v in predictions.items() if k.endswith('_d')}
a_prediction=predictions['semimajor_geometric_km']
a_percent_error=100*abs(a_prediction-reference_heldout['a_km'])/reference_heldout['a_km']
check('truth_only_revealed_in_phase_two', reference_heldout['name'] not in observed)
check('geometric_model_beats_arithmetic_midpoint', period_errors['geometric_bridge_d']['absolute_error_d']<period_errors['arithmetic_midpoint_d']['absolute_error_d'])
check('a_prediction_between_observed_a',observed['Io']['a_km']<a_prediction<observed['Ganymede']['a_km'])
print('REVEALED Europa period:',truth,'days')
for model,results in period_errors.items():
    print(f'{model:26s}: {results["predicted_d"]:.8f} d, error={results["percent_error"]:.4f}%')
print('Europa semi-major-axis geometric estimate:',round(a_prediction,2),'km; error=',round(a_percent_error,4),'%')

REVEALED Europa period: 3.551181 days
geometric_bridge_d        : 3.55772281 d, error=0.1842%
inner_times_two_d         : 3.53827600 d, error=0.3634%
outer_divided_two_d       : 3.57727650 d, error=0.7348%
arithmetic_midpoint_d     : 4.46184550 d, error=25.6440%
Europa semi-major-axis geometric estimate: 671933.57 km; error= 0.1242 %


## LEVEL 4 — Scientific verdict and research handoff

**Scope of PASS:** math / code reproducibility of a known-resonance-period reconstruction under explicit assumptions. The withheld fact is observationally well established, and model-family knowledge of the resonance is *not blind*. Therefore the result does **not** validate new CHH physical laws, a new body detector, Einstein–æther coupling, or GVH Dynamic.

**What is open:** acquire independent dynamical traces (astrometric perturbations, transit-time variations / resonant-angle time series); compare model `H0` vs `H1` vs `H2` with complexity penalization and physical constraints; keep a whole planetary or moon system held out. Test whether one can distinguish a lone hidden subcube from multiple hidden subcubes. Reconstructing an orbital period is not reconstructing mass or other hidden properties.

**Frozen boundary:** `GVH_USED=False` for ALMA (separate branch); no `EOM-GVH`; CHH intrinsically non-derivative.\
**Next proposed gate:** `GVH_CHH_Adjacent_Orbital_Independent_Dynamical_Evidence_Gate_FAST` (requires time-resolved data and comparison with Newtonian N-body model).


In [5]:
summary={
 'notebook':NAME, 'protocol':'GVH_CHH_PROTOCOL_FROZEN_v1.0',
 'execution_status':'PASS_RETROSPECTIVE_PERIOD_RECONSTRUCTION_ONLY',
 'body_detection_status':'ABSTAIN_BODY_EXISTENCE',
 'physical_validation_status':'OPEN_LOCK_NO_NEW_LAW',
 'source_urls':[SOURCE_URL,RES_URL],
 'claim_scope':'Known Jovian resonance-based masked reference check; not discovery, not independent heldout-system generalization',
 'pre_reveal_sha256':pre_reveal_sha256,
 'source_observed':observed,'heldout_reference':reference_heldout,
 'predictions':predictions,'percent_period_errors':period_errors,
 'semimajor_geometric_percent_error':a_percent_error,
 'null_gap_ratio':null_ratio,'synthetic_null_four_to_one_trigger':synthetic_gate,
 'derivative_label':'AUX-SENSITIVITY','EOM_GVH':False,'GVH_USED':False,
 'method_is_universal':False,'sqrt22_tested':False,
 'checks':checks,'checks_total':len(checks),'checks_passed':sum(c['pass'] for c in checks),
 'next_gate':'GVH_CHH_Adjacent_Orbital_Independent_Dynamical_Evidence_Gate_FAST'
}
check('check_counts_consistent',summary['checks_total']==summary['checks_passed'])
summary['checks']=checks
summary['checks_total']=len(checks)
summary['checks_passed']=sum(c['pass'] for c in checks)
assert summary['checks_total']==summary['checks_passed']
results_path=ROOT/(NAME+'_results.json')
results_path.write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
print('STATUS',summary['execution_status'],';',summary['body_detection_status'])
print('CHECKS',summary['checks_passed'],'/',summary['checks_total'])
print('Results',results_path)

STATUS PASS_RETROSPECTIVE_PERIOD_RECONSTRUCTION_ONLY ; ABSTAIN_BODY_EXISTENCE
CHECKS 16 / 16
Results /mnt/data/GVH_CHH_Adjacent_Orbital_Subcube_Blind_Reconstruction_Gate_FAST_results.json
